In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 重新读取原始数据
df = pd.read_csv("/content/cs-training.csv")
df = df.rename(columns={"Unnamed: 0": "customer_id"})

target = "SeriousDlqin2yrs"

# 数据完整性检查：目标变量不允许缺失
assert df.shape[0] == 150000, "数据行数不是150000，请检查文件"
assert df[target].isna().sum() == 0, "目标变量存在缺失，请检查数据"

# 处理明确异常值
df.loc[df["age"] == 0, "age"] = np.nan

delinquency_columns = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate"
]

for column in delinquency_columns:
    df[column] = df[column].replace([96, 98], np.nan)

# 分离特征和目标
X = df.drop(columns=["customer_id", target])
y = df[target].astype(int)

# 划分数据
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print("Training default rate:", y_train.mean())
print("Test default rate:", y_test.mean())

Training set: (120000, 10)
Test set: (30000, 10)
Training default rate: 0.06684166666666666
Test default rate: 0.06683333333333333


In [10]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

preprocessor = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),
    (
        "scaler",
        StandardScaler()
    )
])

In [11]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training set:", X_train_processed.shape)
print("Processed test set:", X_test_processed.shape)

Processed training set: (120000, 16)
Processed test set: (30000, 16)


In [12]:
print(
    "Missing values in processed training set:",
    np.isnan(X_train_processed).sum()
)

print(
    "Missing values in processed test set:",
    np.isnan(X_test_processed).sum()
)

Missing values in processed training set: 0
Missing values in processed test set: 0


## Preprocessing Summary

- Removed the customer identifier from the modelling features.
- Replaced an invalid age of zero with a missing value.
- Treated delinquency values 96 and 98 as anomalous values.
- Split the labelled data into stratified training and test sets.
- Used training-set medians to impute missing numerical values.
- Added missing-value indicators.
- Standardised numerical features.
- Applied preprocessing rules learned from the training set to the test set to prevent data leakage.

## Next Step

Build an interpretable logistic regression baseline using the preprocessing pipeline.